# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset object
dataset = mlc.Dataset(croissant_url)
# Access the metadata
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'unknown')}")
print(f"License: {getattr(metadata, 'license', 'unknown')}")
print(f"Fields: {getattr(metadata, 'keywords', 'unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities including RecordSets, Fields, and Columns are referenced by their `@id` as per the Croissant schema.

In [ ]:
# List all RecordSets in the dataset, referencing them by '@id'
record_sets = dataset.record_sets
print(f"Record Sets found ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}\n")
    # List fields in this record set (by field @id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, Type: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, data from each record set will be loaded. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all RecordSet @id's
record_set_ids = [rs.id for rs in record_sets]
# Load records for each RecordSet into a DataFrame, keyed by RecordSet @id
dataframes = {}

for record_set in record_sets:
    rs_id = record_set.id
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {df.shape[0]} records for RecordSet @id: {rs_id}\nColumns:@id: {df.columns.tolist()}\n")
    else:
        print(f"No records loaded for RecordSet @id: {rs_id}")

# For demonstration, pick the first non-empty record set
main_rs_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_rs_id = rsid
        break
if main_rs_id is not None:
    print(f"Displaying first 5 records for RecordSet: {main_rs_id}")
    display(dataframes[main_rs_id].head())
else:
    print("No non-empty RecordSets to display.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All field references below use their Croissant schema `@id` (column names in the DataFrame).

In [ ]:
# We'll run EDA on the main record set (main_rs_id)
import numpy as np

if main_rs_id is not None:
    df = dataframes[main_rs_id]
    # Find candidate numeric columns (float or int types or likely numeric by name)
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or any(s in col.lower() for s in ['mw', 'weight', 'abundance', 'count', 'peptide', 'coverage', 'ratio', 'number', 'intensity'])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field} (@id) for filtering and normalization.")
    else:
        print("No numeric field found for EDA.")
        numeric_field = None

    if numeric_field is not None:
        # Convert to numeric if not already
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)  # e.g., upper quartile as filter threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Number of records with {numeric_field} > {threshold:.2f}: {len(filtered_df)}\n")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field (e.g. field that contains 'type', 'group', 'category', etc.)
        group_fields = [col for col in df.columns if any(token in col.lower() for token in ['type','group','sample','category','class'])]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by {group_field} (@id):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No group field found for aggregation.")
else:
    print("No suitable DataFrame for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the selected numeric field and (if available) its grouping by a top category/group field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and numeric_field is not None and not filtered_df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in filtered records")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If group_field was found
    if 'group_field' in locals():
        plt.figure(figsize=(9,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No sufficient numeric data for visualization.")

## 6. Conclusion

This notebook provided an overview and exploration of the FAIR^2 dataset _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_ using the `mlcroissant` library.

- The metadata provided context and allowed programmatic extraction of fields and record sets using their `@id`.
- We loaded records from available record sets, performed basic filtering and normalization using numeric fields, and visualized key data distributions.
- For further analysis, you may join the record sets, combine fields, or use domain-specific techniques appropriate for proteomics mass spec data.

_Remember to always refer to Croissant entities by their `@id` for reproducible, schema-driven programmatic workflows!_
